In [1]:
# 06 - Final Model Evaluation
# Compares every model variant on both targets with the same split and folds,
# checks generalisation (train vs CV vs test) and records the final selection.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parents[1])
print("Working directory:", Path.cwd())

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBRegressor

Working directory: C:\Users\shame\OneDrive\Desktop\EcoPackAI


In [2]:
materials = pd.read_csv("data/processed/materials.csv")
train_df, test_df = train_test_split(materials, test_size=0.2, random_state=42)

preprocessor = joblib.load("models/preprocessing/preprocessing_pipeline.pkl")
X_train = preprocessor.transform(train_df)
X_test = preprocessor.transform(test_df)
FOLDS = KFold(n_splits=5)   # the same 5 folds every earlier experiment used
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (322, 31) Test: (81, 31)


In [3]:
def scores(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": root_mean_squared_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred)}


def evaluate_model(model, target):
    """Train, 5-fold CV and test scores for a scikit-learn style model."""
    cv = cross_validate(clone(model), X_train, train_df[target], cv=FOLDS,
                        scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"])
    fitted = clone(model).fit(X_train, train_df[target])
    return {
        "train": scores(train_df[target], fitted.predict(X_train)),
        "cv": {"MAE": -cv["test_neg_mean_absolute_error"].mean(),
               "RMSE": -cv["test_neg_root_mean_squared_error"].mean(),
               "R2": cv["test_r2"].mean()},
        "test": scores(test_df[target], fitted.predict(X_test)),
    }


def evaluate_lookup(target):
    """Naive reference: predict the training average of the material type."""
    def fit_predict(fit_rows, predict_rows):
        means = fit_rows.groupby("material_type")[target].mean()
        return predict_rows["material_type"].map(means)

    folds = [scores(train_df.iloc[b][target], fit_predict(train_df.iloc[a], train_df.iloc[b]))
             for a, b in FOLDS.split(train_df)]
    return {
        "train": scores(train_df[target], fit_predict(train_df, train_df)),
        "cv": {m: np.mean([f[m] for f in folds]) for m in ["MAE", "RMSE", "R2"]},
        "test": scores(test_df[target], fit_predict(train_df, test_df)),
    }

In [4]:
rf_cost = joblib.load("ml/models/rf_cost.joblib").named_steps["model"]
xgb_co2 = joblib.load("ml/models/xgb_co2.joblib").named_steps["model"]

VARIANTS = {
    "cost_per_unit_usd": {
        "Material-type average": None,
        "Linear Regression (baseline v1)": joblib.load("ml/models/LinearRegression_cost_per_unit_usd_v1.pkl"),
        "Random Forest (production v1)": rf_cost,
        "XGBoost (alternative, CO2 settings)": clone(xgb_co2),
    },
    "co2_emission_per_kg": {
        "Material-type average": None,
        "Decision Tree (baseline v1)": joblib.load("ml/models/DecisionTreeRegressor_co2_emission_per_kg_v1.pkl"),
        "XGBoost (production v1)": xgb_co2,
        "Random Forest (alternative, cost settings)": clone(rf_cost),
    },
}

rows = []
for target, models in VARIANTS.items():
    for name, model in models.items():
        result = evaluate_lookup(target) if model is None else evaluate_model(model, target)
        for split, values in result.items():
            for metric, value in values.items():
                rows.append({"target": target, "model": name, "split": split, "metric": metric, "value": value})

comparison = pd.DataFrame(rows)
table = comparison.pivot_table(index=["target", "model"], columns=["split", "metric"], values="value", sort=False)
table.round(4)

split                                                            train  \
metric                                                             MAE   
target              model                                                
cost_per_unit_usd   Material-type average                       1.8816   
                    Linear Regression (baseline v1)             0.3184   
                    Random Forest (production v1)               0.0854   
                    XGBoost (alternative, CO2 settings)         0.2119   
co2_emission_per_kg Material-type average                       0.0398   
                    Decision Tree (baseline v1)                 0.0001   
                    XGBoost (production v1)                     0.0291   
                    Random Forest (alternative, cost settings)  0.0159   

split                                                                   \
metric                                                            RMSE   
target              model                                                
cost_per_unit_usd   Material-type average                       3.3816   
                    Linear Regression (baseline v1)             0.3936   
                    Random Forest (production v1)               0.1923   
                    XGBoost (alternative, CO2 settings)         0.2705   
co2_emission_per_kg Material-type average                       0.0589   
                    Decision Tree (baseline v1)                 0.0012   
                    XGBoost (production v1)                     0.0390   
                    Random Forest (alternative, cost settings)  0.0243   

split                                                                   \
metric                                                              R2   
target              model                                                
cost_per_unit_usd   Material-type average                       0.7748   
                    Linear Regression (baseline v1)             0.9969   
                    Random Forest (production v1)               0.9993   
                    XGBoost (alternative, CO2 settings)         0.9986   
co2_emission_per_kg Material-type average                       0.9935   
                    Decision Tree (baseline v1)                 1.0000   
                    XGBoost (production v1)                     0.9971   
                    Random Forest (alternative, cost settings)  0.9989   

split                                                               cv  \
metric                                                             MAE   
target              model                                                
cost_per_unit_usd   Material-type average                       1.9058   
                    Linear Regression (baseline v1)             0.3567   
                    Random Forest (production v1)               0.2466   
                    XGBoost (alternative, CO2 settings)         0.3575   
co2_emission_per_kg Material-type average                       0.0405   
                    Decision Tree (baseline v1)                 0.0546   
                    XGBoost (production v1)                     0.0489   
                    Random Forest (alternative, cost settings)  0.0451   

split                                                                   \
metric                                                            RMSE   
target              model                                                
cost_per_unit_usd   Material-type average                       3.3551   
                    Linear Regression (baseline v1)             0.4420   
                    Random Forest (production v1)               0.4796   
                    XGBoost (alternative, CO2 settings)         0.6112   
co2_emission_per_kg Material-type average                       0.0591   
                    Decision Tree (baseline v1)                 0.0823   
                    XGBoost (production v1)                     0.0833   


In [5]:
# Generalisation: how much worse is unseen data than the data the model trained on?
gaps = table.xs("RMSE", axis=1, level="metric")[["train", "cv", "test"]].copy()
gaps["test / train RMSE"] = gaps["test"] / gaps["train"]
gaps.round(4)

split                                                            train  \
target              model                                                
cost_per_unit_usd   Material-type average                       3.3816   
                    Linear Regression (baseline v1)             0.3936   
                    Random Forest (production v1)               0.1923   
                    XGBoost (alternative, CO2 settings)         0.2705   
co2_emission_per_kg Material-type average                       0.0589   
                    Decision Tree (baseline v1)                 0.0012   
                    XGBoost (production v1)                     0.0390   
                    Random Forest (alternative, cost settings)  0.0243   

split                                                               cv  \
target              model                                                
cost_per_unit_usd   Material-type average                       3.3551   
                    Linear Regression (baseline v1)             0.4420   
                    Random Forest (production v1)               0.4796   
                    XGBoost (alternative, CO2 settings)         0.6112   
co2_emission_per_kg Material-type average                       0.0591   
                    Decision Tree (baseline v1)                 0.0823   
                    XGBoost (production v1)                     0.0833   
                    Random Forest (alternative, cost settings)  0.0694   

split                                                             test  \
target              model                                                
cost_per_unit_usd   Material-type average                       2.8433   
                    Linear Regression (baseline v1)             0.4044   
                    Random Forest (production v1)               0.3101   
                    XGBoost (alternative, CO2 settings)         0.3768   
co2_emission_per_kg Material-type average                       0.0503   
                    Decision Tree (baseline v1)                 0.0780   
                    XGBoost (production v1)                     0.0537   
                    Random Forest (alternative, cost settings)  0.0513   

split                                                           test / train RMSE  
target              model                                                          
cost_per_unit_usd   Material-type average                                  0.8408  
                    Linear Regression (baseline v1)                        1.0273  
                    Random Forest (production v1)                          1.6126  
                    XGBoost (alternative, CO2 settings)                    1.3930  
co2_emission_per_kg Material-type average                                  0.8549  
                    Decision Tree (baseline v1)                           65.9469  
                    XGBoost (production v1)                                1.3792  
                    Random Forest (alternative, cost settings)             2.1114

In [6]:
SELECTED = {"cost_per_unit_usd": "Random Forest (production v1)",
            "co2_emission_per_kg": "XGBoost (production v1)"}

for target, chosen in SELECTED.items():
    test_rmse = table.loc[target].xs(("test", "RMSE"), axis=1)
    cv_rmse = table.loc[target].xs(("cv", "RMSE"), axis=1)
    print(f"{target}: selected {chosen}")
    print(f"   lowest test RMSE: {test_rmse.idxmin()} ({test_rmse.min():.4f}); selected {test_rmse[chosen]:.4f}")
    print(f"   lowest CV RMSE:   {cv_rmse.idxmin()} ({cv_rmse.min():.4f}); selected {cv_rmse[chosen]:.4f}")

cost_per_unit_usd: selected Random Forest (production v1)
   lowest test RMSE: Random Forest (production v1) (0.3101); selected 0.3101
   lowest CV RMSE:   Linear Regression (baseline v1) (0.4420); selected 0.4796
co2_emission_per_kg: selected XGBoost (production v1)
   lowest test RMSE: Material-type average (0.0503); selected 0.0537
   lowest CV RMSE:   Material-type average (0.0591); selected 0.0833


In [7]:
out = comparison.pivot_table(index=["target", "model"], columns=["split", "metric"], values="value", sort=False)
out.columns = [f"{split}_{metric}" for split, metric in out.columns]
out = out.reset_index()
out["selected"] = out.apply(lambda r: SELECTED[r["target"]] == r["model"], axis=1)
out.round(6).to_csv("ml/metrics/final_model_comparison.csv", index=False)
out[["target", "model", "test_MAE", "test_RMSE", "test_R2", "cv_RMSE", "selected"]].round(4)

,target,model,test_MAE,test_RMSE,test_R2,cv_RMSE,selected
0,cost_per_unit_usd,Material-type average,1.5484,2.8433,0.8306,3.3551,False
1,cost_per_unit_usd,Linear Regression (baseline v1),0.3329,0.4044,0.9966,0.4420,False
2,cost_per_unit_usd,Random Forest (production v1),0.1888,0.3101,0.9980,0.4796,True
3,cost_per_unit_usd,"XGBoost (alternative, CO2 settings)",0.2820,0.3768,0.9970,0.6112,False
4,co2_emission_per_kg,Material-type average,0.0333,0.0503,0.9945,0.0591,False
5,co2_emission_per_kg,Decision Tree (baseline v1),0.0507,0.0780,0.9869,0.0823,False
6,co2_emission_per_kg,XGBoost (production v1),0.0358,0.0537,0.9938,0.0833,True
7,co2_emission_per_kg,"Random Forest (alternative, cost settings)",0.0350,0.0513,0.9943,0.0694,False
